In [ ]:
%%writefile omp_tp.c
#include <stdio.h>
#include <omp.h>

int main() {
    int n, num_threads;

    printf("Enter number of threads: ");
    scanf("%d", &num_threads);

    printf("Enter number of iterations: ");
    scanf("%d", &n);

    omp_set_num_threads(num_threads);  //here we r setting the number of threads

    long long sum = 0;

    double start = omp_get_wtime();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        int total = omp_get_num_threads();

        #pragma omp for schedule(static) reduction(+:sum) //to add static split with creating sum for each thread
        for (int i = 0; i < n; i++) {
            printf("Thread %d / %d -> iteration %d\n", tid, total, i);
            sum += i;
        }
    }

    double end = omp_get_wtime();

    printf("\nFinal sum = %lld\n", sum);
    printf("Execution time = %f seconds\n", end - start);

    return 0;
}

Writing omp_tp.c


In [ ]:
!gcc -fopenmp omp_tp.c -o omp_tp

In [ ]:
!echo -e "4\n100" | ./omp_tp

Enter number of threads: Enter number of iterations: Thread 2 / 4 -> iteration 50
Thread 3 / 4 -> iteration 75
Thread 1 / 4 -> iteration 25
Thread 1 / 4 -> iteration 26
Thread 0 / 4 -> iteration 0
Thread 0 / 4 -> iteration 1
Thread 0 / 4 -> iteration 2
Thread 0 / 4 -> iteration 3
Thread 0 / 4 -> iteration 4
Thread 0 / 4 -> iteration 5
Thread 0 / 4 -> iteration 6
Thread 0 / 4 -> iteration 7
Thread 0 / 4 -> iteration 8
Thread 0 / 4 -> iteration 9
Thread 0 / 4 -> iteration 10
Thread 0 / 4 -> iteration 11
Thread 0 / 4 -> iteration 12
Thread 0 / 4 -> iteration 13
Thread 0 / 4 -> iteration 14
Thread 0 / 4 -> iteration 15
Thread 0 / 4 -> iteration 16
Thread 0 / 4 -> iteration 17
Thread 0 / 4 -> iteration 18
Thread 0 / 4 -> iteration 19
Thread 0 / 4 -> iteration 20
Thread 0 / 4 -> iteration 21
Thread 0 / 4 -> iteration 22
Thread 0 / 4 -> iteration 23
Thread 0 / 4 -> iteration 24
Thread 1 / 4 -> iteration 27
Thread 3 / 4 -> iteration 76
Thread 3 / 4 -> iteration 77
Thread 1 / 4 -> iteration 28


OpenMP divides a task into parts and assigns each part to a different thread. These threads
 run at the same time, allowing the program to execute faster.

THE SUM A-appearing in the final is a combined sum of the 4 sums created by each thread ...............that has been created because of reduction


**DYNAMIC**

In [ ]:
%%writefile omp_dynamic.c
#include <stdio.h>
#include <omp.h>

int main() {
    int n, num_threads;

    printf("Enter number of threads: ");
    scanf("%d", &num_threads);

    printf("Enter number of iterations: ");
    scanf("%d", &n);

    omp_set_num_threads(num_threads);  // set number of threads

    long long sum = 0;

    double start = omp_get_wtime();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        int total = omp_get_num_threads();


        #pragma omp for schedule(dynamic, 10) reduction(+:sum) //that 10 means a thread takes 10 iterations when it finish it can take 10 others
        for (int i = 0; i < n; i++) {
            printf("Thread %d / %d -> iteration %d\n", tid, total, i);
            sum += i;
        }
    }

    double end = omp_get_wtime();

    printf("\nFinal sum = %lld\n", sum);
    printf("Execution time = %f seconds\n", end - start);

    return 0;
}

Overwriting omp_dynamic.c


In [ ]:
!gcc -fopenmp omp_dynamic.c -o omp_dynamic

In [ ]:
!./omp_dynamic 4 100

Enter number of threads: 4
Enter number of iterations: 100
Thread 1 / 4 -> iteration 10
Thread 1 / 4 -> iteration 11
Thread 1 / 4 -> iteration 12
Thread 1 / 4 -> iteration 13
Thread 1 / 4 -> iteration 14
Thread 1 / 4 -> iteration 15
Thread 1 / 4 -> iteration 16
Thread 1 / 4 -> iteration 17
Thread 2 / 4 -> iteration 20
Thread 0 / 4 -> iteration 30
Thread 0 / 4 -> iteration 31
Thread 0 / 4 -> iteration 32
Thread 0 / 4 -> iteration 33
Thread 0 / 4 -> iteration 34
Thread 0 / 4 -> iteration 35
Thread 0 / 4 -> iteration 36
Thread 0 / 4 -> iteration 37
Thread 0 / 4 -> iteration 38
Thread 0 / 4 -> iteration 39
Thread 1 / 4 -> iteration 18
Thread 2 / 4 -> iteration 21
Thread 2 / 4 -> iteration 22
Thread 2 / 4 -> iteration 23
Thread 2 / 4 -> iteration 24
Thread 2 / 4 -> iteration 25
Thread 2 / 4 -> iteration 26
Thread 2 / 4 -> iteration 27
Thread 2 / 4 -> iteration 28
Thread 2 / 4 -> iteration 29
Thread 2 / 4 -> iteration 50
Thread 2 / 4 -> iteration 51
Thread 2 / 4 -> iteration 52
Thread 2 / 4 

🟨 2. DYNAMIC scheduling
🧠 Idea:

Work is given during execution

⚙️ Behavior:
Threads take small chunks (e.g., 10 iterations)
When a thread finishes, it takes more work

DYNAMIC is faster because Threads take new work when they finish.


**Guided**

In [ ]:
%%writefile omp_guided.c
#include <stdio.h>
#include <omp.h>

int main() {
    int n, num_threads;

    printf("Enter number of threads: ");
    scanf("%d", &num_threads);

    printf("Enter number of iterations: ");
    scanf("%d", &n);

    omp_set_num_threads(num_threads);  // set number of threads

    long long sum = 0;

    double start = omp_get_wtime();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        int total = omp_get_num_threads();

        // 🔥 CHANGED: guided scheduling
        #pragma omp for schedule(guided) reduction(+:sum)
        for (int i = 0; i < n; i++) {
            printf("Thread %d / %d -> iteration %d\n", tid, total, i);
            sum += i;
        }
    }

    double end = omp_get_wtime();

    printf("\nFinal sum = %lld\n", sum);
    printf("Execution time = %f seconds\n", end - start);

    return 0;
}

Writing omp_guided.c


In [ ]:
!gcc -fopenmp omp_guided.c -o omp_guided

In [ ]:
!./omp_guided

Enter number of threads: 4
Enter number of iterations: 100
Thread 1 / 4 -> iteration 25
Thread 1 / 4 -> iteration 26
Thread 1 / 4 -> iteration 27
Thread 1 / 4 -> iteration 28
Thread 1 / 4 -> iteration 29
Thread 1 / 4 -> iteration 30
Thread 1 / 4 -> iteration 31
Thread 1 / 4 -> iteration 32
Thread 1 / 4 -> iteration 33
Thread 1 / 4 -> iteration 34
Thread 1 / 4 -> iteration 35
Thread 1 / 4 -> iteration 36
Thread 0 / 4 -> iteration 58
Thread 0 / 4 -> iteration 59
Thread 0 / 4 -> iteration 60
Thread 0 / 4 -> iteration 61
Thread 0 / 4 -> iteration 62
Thread 0 / 4 -> iteration 63
Thread 0 / 4 -> iteration 64
Thread 2 / 4 -> iteration 44
Thread 0 / 4 -> iteration 65
Thread 0 / 4 -> iteration 66
Thread 0 / 4 -> iteration 67
Thread 0 / 4 -> iteration 68
Thread 0 / 4 -> iteration 69
Thread 0 / 4 -> iteration 70
Thread 0 / 4 -> iteration 71
Thread 0 / 4 -> iteration 72
Thread 0 / 4 -> iteration 73
Thread 0 / 4 -> iteration 74
Thread 0 / 4 -> iteration 75
Thread 0 / 4 -> iteration 76
Thread 0 / 4 

The same idea as dynamic but shunch are not with constant size their size become small over


⚙️ How it works:
First: large chunks assigned
Later: smaller chunks
Threads finish faster and steal smaller tasks

**EXERCICE 2**

In [ ]:
%%writefile omp_ex2.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

#define N 20

int main() {

    int arr[N];
    int sum = 0;

    omp_set_num_threads(4);

    #pragma omp parallel     //Start multiple threads here, and each thread executes this block
    {
        int tid = omp_get_thread_num();

        // 1. Only one thread initializes array
        #pragma omp single
        {
            printf("Thread %d initializing array...\n", tid);
            for (int i = 0; i < N; i++) {
                arr[i] = rand() % 10;
            }
        }

        // 2. Wait for initialization
        #pragma omp barrier  //All threads must wait here

        // 3. Critical section for synchronized messages
        #pragma omp critical   //Only ONE thread at a time can enter this block
        {
            printf("Thread %d starting computation\n", tid);
        }

        // 4. Parallel sum using atomic
        #pragma omp for       //Split the loop between all threads
        for (int i = 0; i < N; i++) {
            #pragma omp atomic    //Only ONE thread at a time can update sum
            sum += arr[i];
        }

        // 5. Barrier before final output
        #pragma omp barrier     //all threads must wait untill all finish their calculations

        // 6. Master prints final result
        #pragma omp master        //is used so that not all threads print the result. only master thread[0]
        {
            printf("\nFinal sum = %d\n", sum);
        }
    }

    return 0;
}

Writing omp_ex2.c


In [ ]:
!gcc -fopenmp omp_ex2.c -o omp_ex2

In [ ]:
!./omp_ex2

Thread 1 initializing array...
Thread 1 starting computation
Thread 3 starting computation
Thread 0 starting computation
Thread 2 starting computation

Final sum = 88


🧠 KEY IDEA

Threads are created for the whole region, not for a single task.

We create all threads at the start because OpenMP builds a reusable thread team for the entire parallel region, even if only one thread performs the first task.

**EXERCICE 3**

In [ ]:
%%writefile omp_ex3.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main() {

    int n, num_threads;

    printf("Enter number of threads: ");
    scanf("%d", &num_threads);

    printf("Enter matrix size N: ");
    scanf("%d", &n);

    omp_set_num_threads(num_threads);

    // Enable nested parallelism
    omp_set_nested(1);   //side a thread you can create another parallel region//1 MA3ANAH ENABLE NESTED PARALLISM

    int matrix[n][n];
    long long sum = 0;

    // 1. Initialize matrix (single thread)
    #pragma omp parallel
    {
        #pragma omp single
        {
            printf("Initializing matrix...\n");
            for (int i = 0; i < n; i++) {
                for (int j = 0; j < n; j++) {
                    matrix[i][j] = rand() % 10;
                }
            }
        }

        #pragma omp barrier

        // 2. Parallel nested loops for sum
        #pragma omp for reduction(+:sum)
        for (int i = 0; i < n; i++) {

            // nested parallelism (inner loop)
            #pragma omp parallel for reduction(+:sum)
            for (int j = 0; j < n; j++) {
                sum += matrix[i][j];
            }
        }
    }

    printf("\nFinal sum = %lld\n", sum);

    return 0;
}

Writing omp_ex3.c


🧠 Nested parallelism means a thread can create another parallel region where a new group of threads is spawned — not that one thread directly becomes multiple threads.